# 🚀 Lab 01 — Initiation à GitHub Actions (Mode Tutoriel)

Bienvenue dans ce lab inspiré des tutoriels *GitHub Learning Lab* ! 

Avant de construire des pipelines complexes de données, il faut comprendre le moteur qui va les faire tourner : **GitHub Actions**.

**Objectifs pédagogiques :**
- Découvrir la CI/CD (Intégration et Déploiement Continus).
- Comprendre l'architecture d'un Workflow (`.github/workflows/*.yml`).
- Manipuler les déclencheurs (`on: push`, `schedule`, `workflow_dispatch`).
- Gérer un environnement (ex: installer Python sur un serveur distant).
- Exécuter du code SQL de manière automatisée.
- Utiliser les Secrets pour cacher des mots de passe.

---

## 🛠️ Partie 1 : Votre Premier Workflow "Hello World"

GitHub Actions permet d'exécuter du code sur des serveurs fournis par GitHub (appelés *runners*).
Tout commence par un fichier de configuration écrit en **YAML**.

### 1.1 Préparer le terrain
1. Allez sur [github.com](https://github.com) et créez un nouveau dépôt public nommé `learning-github-actions`.
2. Sur votre ordinateur (via le terminal), tapez les commandes indiquez par Github

### 1.2 Créer le workflow YAML
Les workflows GitHub Actions **doivent absolument** être placés dans un dossier spécifique : `.github/workflows/`.

Créez ce dossier et un fichier nommé `hello.yml` :

In [ ]:
mkdir -p .github/workflows
touch .github/workflows/hello.yml

Ouvrez `hello.yml` dans votre éditeur de code et collez le contenu suivant :

```yaml
name: Mon Premier Workflow

# Quand est-ce que ce workflow s'exécute ?
on:
  push:
    branches:
      - main
  workflow_dispatch: # Permet de lancer le workflow manuellement avec un bouton

# Que doit faire ce workflow ?
jobs:
  hello-job:
    # Sur quel type d'ordinateur (Runner) ?
    runs-on: ubuntu-latest
    
    # Quelles sont les étapes ?
    steps:
      - name: Dire bonjour
        run: echo "Hello World! Bienvenue sur GitHub Actions 🚀"
        
      - name: Afficher les infos système
        run: |
          echo "Système d'exploitation :"
          uname -a
          echo "Qui suis-je ?"
          whoami
```

### 1.3 Pousser et Observer
Il est temps d'envoyer (push) ce code sur GitHub !

In [ ]:
git add .
git commit -m "Ajout du workflow Hello World"
git push origin main

**👀 Sur GitHub :**
1. Allez sur la page de votre dépôt sur GitHub.
2. Cliquez sur l'onglet **"Actions"** (en haut, à côté de "Pull requests").
3. Vous devriez voir votre workflow en cours d'exécution (ou terminé avec une coche verte ✅).
4. Cliquez dessus pour lire les logs du terminal distant ! Vous verrez votre "Hello World!".

### Questions: 
- Combiez de temps a duré ce worklow?
- Identifiez les différents étapes

---

## 🐍 Partie 2 : Exécuter un script Python (Environnement)

Dans le monde de la Data, on exécute souvent des scripts Python (comme `dlt` pour l'ingestion). Mais comment dire à la machine GitHub d'installer Python ?

Nous allons utiliser des **Actions pré-existantes** (`uses:`). Ce sont des "blocs de code" prêts à l'emploi créés par la communauté.

### 2.1 Créer un script Python
Créez un fichier `data_script.py` à la racine de votre projet avec le contenu suivant :

In [ ]:
import sys
import datetime

print(f"Version de Python : {sys.version}")
print(f"Date d'exécution : {datetime.datetime.now()}")
print("✅ Traitement des données terminé avec succès.")


### 2.2 Créer le workflow Python
Créez un nouveau fichier `.github/workflows/python_job.yml` :

```yaml
name: Job Python Data

on:
  workflow_dispatch:

jobs:
  run-python:
    runs-on: ubuntu-latest
    steps:
      # Étape 1 : Télécharger notre code sur le runner
      - name: 📥 Récupérer le code
        uses: actions/checkout@v4

      # Étape 2 : Installer Python
      - name: 🐍 Configurer Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'

      # Étape 3 : Exécuter notre script
      - name: 🚀 Lancer le script Data
        run: python data_script.py
```

### 2.3 Pousser et tester
Committez et poussez vos changements.

**👀 Sur GitHub :**
1. Retournez dans l'onglet **"Actions"**.
2. À gauche, cliquez sur **"Job Python Data"**.
3. Puisque nous avons utilisé `workflow_dispatch`, vous verrez un bouton bleu **"Run workflow"** à droite. Cliquez dessus !
4. Observez l'installation de Python et l'exécution de votre script en direct.

---

## 🗄️ Partie 3 : Exécuter un script SQL (Outils pré-installés)

Tout comme on exécute du Python, on peut exécuter du SQL !
Les machines virtuelles (`ubuntu-latest`) de GitHub sont livrées avec de nombreux outils de la Data **déjà pré-installés**, comme le client `bq` (Google BigQuery) ou la base de données légère `sqlite3`.

Dans cet exemple, pour s'entraîner sans avoir besoin de se connecter à Google Cloud, nous allons utiliser `sqlite3`.
Le principe sera **exactement le même** avec `bq` pour l'examen.

### 3.1 Créer un script SQL
Créez un dossier `sql/` et un fichier `transformation.sql` à la racine de votre projet :

In [ ]:
CREATE TABLE etudiants (id INTEGER, prenom TEXT, role TEXT);

INSERT INTO etudiants VALUES (1, 'Alice', 'Data Engineer');
INSERT INTO etudiants VALUES (2, 'Bob', 'Data Scientist');
INSERT INTO etudiants VALUES (3, 'Charlie', 'Data Analyst');

-- Transformation simple
SELECT prenom, role FROM etudiants WHERE role = 'Data Engineer';


### 3.2 Créer le workflow SQL
Créez un nouveau fichier `.github/workflows/sql_job.yml` :


```yaml
name: Job SQL Data

on:
  workflow_dispatch:

jobs:
  run-sql:
    runs-on: ubuntu-latest
    steps:
      - name: Récupérer le code
        uses: actions/checkout@v4

      - name: Exécuter la requête SQL
        # La flèche < permet "d'injecter" le fichier sql directement dans le moteur sqlite3
        # Lors de l'examen, vous remplacerez "sqlite3 test.db" par "bq query" !
        run: sqlite3 base_test.db < sql/transformation.sql
```

### 3.3 Pousser et tester
Committez et poussez vos changements, puis déclenchez le workflow depuis l'onglet **Actions** de GitHub !

Le workflow est-il executé ? Si non, faites les changements nécessaire

### Question:
- Quel est le déclencheur des workflows GitHub Actions ?
- Parcourez la documentation et cherchez 2 autres type de déclencheur 
- Citez un example concret où vous pourriez utiliser celui ci

---

## 🔐 Partie 4 : Les Secrets (Ne jamais mettre de mot de passe dans le code)

Comment se connecter à une base de données ou à une API sans écrire le mot de passe dans le fichier Python ? On utilise les **Secrets GitHub**.

### 4.1 Créer un Secret sur GitHub
1. Sur la page GitHub de votre dépôt, allez dans l'onglet **"Settings"** ⚙️.
2. Dans le menu de gauche, descendez jusqu'à **"Secrets and variables"** -> **"Actions"**.
3. Cliquez sur le bouton vert **"New repository secret"**.
4. **Name** : `MY_SUPER_PASSWORD`
5. **Secret** : `DataIsAwesome2026!`
6. Cliquez sur "Add secret".

### 4.2 Utiliser le secret dans un workflow
Créons un fichier `.github/workflows/secrets_test.yml` :

```yaml
name: Test des Secrets

on: [workflow_dispatch]

jobs:
  print-secret:
    runs-on: ubuntu-latest
    steps:
      - name: Tenter de lire le secret
        # On passe le secret dans une variable d'environnement (env)
        env:
          LE_MOT_DE_PASSE: ${{ secrets.MY_SUPER_PASSWORD }}
        run: |
          echo "Testons l'affichage du mot de passe :"
          echo "Le mot de passe est : $LE_MOT_DE_PASSE"
```

> **Spoiler Alert :** Si vous testez ce workflow, vous verrez que GitHub masque automatiquement le secret dans les logs avec des étoiles `***` pour éviter toute fuite !

---

## 🌩️ Partie 5 : Le lien avec Google Cloud (Workload Identity Federation)

Dans vos vrais pipelines Data (et pour votre examen de cet après-midi), vous aurez besoin d'exécuter des commandes BigQuery (`bq query`).
Plutôt que d'utiliser des mots de passe (clés JSON Service Account) qui peuvent fuiter, la norme de sécurité moderne s'appelle **Workload Identity Federation (WIF)**.

**Le concept :**
GitHub et Google Cloud "se parlent". Google fait confiance à GitHub si ce dernier prouve que le workflow vient bien de votre dépôt.

### Exemple de bloc d'authentification WIF (Pour information - Pas besoin de l'exécuter)

Pour que cela fonctionne, il suffit d'utiliser l'Action officielle de Google (`google-github-actions/auth`) et de demander à GitHub un jeton temporaire (`permissions: id-token: write`).

```yaml
jobs:
  gcp-job:
    runs-on: ubuntu-latest
    
    # ⚠️ Indispensable pour s'authentifier via WIF
    permissions:
      contents: 'read'
      id-token: 'write'

    steps:
      - name: 📥 Récupérer le code
        uses: actions/checkout@v4

      - name: 🔐 Authentification Google Cloud (WIF)
        uses: google-github-actions/auth@v2
        with:
          # Vous n'aurez qu'à copier/coller ces infos fournies par le formateur
          workload_identity_provider: 'projects/833312929773/locations/global/workloadIdentityPools/github-pool/providers/github-provider'
          service_account: 'github-actions-elt@lasalle-big-data.iam.gserviceaccount.com'

      - name: 🚀 Lancer une requête BigQuery
        run: bq query --use_legacy_sql=false < sql/transformation.sql
```

---

## 🧗‍♀️ Partie 6 : À vous de jouer ! (Exercice autonome)

Maintenant que vous avez compris le principe, vous allez créer de bout en bout un nouveau pipeline pour des **commandes (orders)**.

### 6.1 Préparer les données brutes
Exécutez la cellule ci-dessous pour créer le fichier d'initialisation de la table `orders` avec quelques données de test :

In [ ]:
cat << 'EOF' > sql/init_orders.sql
CREATE TABLE IF NOT EXISTS orders (order_id INTEGER, status TEXT, amount REAL);
INSERT INTO orders VALUES (101, 'COMPLETED', 150.50);
INSERT INTO orders VALUES (102, 'CANCELLED', 200.00);
INSERT INTO orders VALUES (103, 'COMPLETED', 99.99);
INSERT INTO orders VALUES (104, 'PENDING',   50.00);
EOF

# Création de la table dans notre petite base de test locale
sqlite3 base_test.db < sql/init_orders.sql
echo "✅ Fichier d'initialisation et table orders créés !"


### 6.2 Votre mission

1. **Créer le fichier de transformation SQL**
   - Créez un nouveau fichier `sql/clean_orders.sql` dans votre projet.
   - Écrivez-y votre propre code SQL : par exemple, une requête qui sélectionne **uniquement les commandes dont le statut est `'COMPLETED'`**.

2. **Créer le workflow GitHub Actions**
   - Créez un nouveau fichier `.github/workflows/orders_pipeline.yml`.
   - Inspirez-vous du workflow YAML de la partie 3.
   - **Attention** : Sur le serveur GitHub, la base de données est vide au démarrage. Vous devez donc ajouter **deux étapes (`run`)** :
     - Une première pour initialiser les données : `sqlite3 base_test.db < sql/init_orders.sql`
     - Une seconde pour exécuter vos transformations : `sqlite3 base_test.db < sql/clean_orders.sql`

3. **Déployer et tester**
   - Committez ces nouveaux fichiers et poussez-les sur votre dépôt GitHub.
   - Allez dans l'onglet **Actions**, lancez votre nouveau workflow manuellement (`workflow_dispatch`), et vérifiez dans les logs que vos commandes filtrées s'affichent correctement !



---
### 🎓 Résumé de ce que vous avez appris
1. **GitHub Actions** est un outil de CI/CD Serverless (des machines virtuelles gratuites à la demande).
2. Les workflows sont définis en **YAML** dans `.github/workflows/`.
3. Les déclencheurs (`on:`) peuvent être des pushs de code, des heures précises (`schedule`), ou manuels (`workflow_dispatch`).
4. On peut utiliser des actions pré-packagées (`uses: actions/setup-python`) pour installer des outils.
5. On peut utiliser la **redirection d'entrée** (`<`) pour injecter un fichier SQL dans un moteur de base de données en ligne de commande.
6. Les **Secrets** protègent les données sensibles.
7. Le **WIF (Workload Identity Federation)** est la manière moderne et sécurisée de connecter GitHub à Google Cloud Platform sans mot de passe statique.

**Prêts pour l'orchestration de votre pipeline ELT complet ?** 🚀